# 01. Batch Processing - Medallion Architecture

This notebook walks through **Bronze → Silver → Gold** layers using NYC Taxi data: raw ingestion to Delta on MinIO, cleaning and enrichment, then aggregations for analytics and downstream ML.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import boto3
import matplotlib.pyplot as plt
import pandas as pd

# Spark / Delta session created in next cell


In [ ]:
# MinIO/S3-style paths — adjust endpoint, bucket, and credentials for your environment
MINIO_ENDPOINT = "http://localhost:9000"
BRONZE_PATH = "s3a://lakehouse/bronze/nyc_taxi/"
SILVER_PATH = "s3a://lakehouse/silver/nyc_taxi/"
GOLD_PATH = "s3a://lakehouse/gold/"

spark = (
    SparkSession.builder.appName("AIDE2-Medallion")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # Replace with your MinIO keys (or use IAM / env-based config in production)
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


## Bronze Layer - Raw Data Ingestion


In [ ]:
RAW_PARQUET = "s3a://raw/nyc-taxi/yellow/*.parquet"  # or local path for dev

bronze_df = spark.read.parquet(RAW_PARQUET)
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

bronze_df = (
    bronze_df.withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_system", F.lit("nyc_tlc_parquet"))
    .withColumn("batch_id", F.monotonically_increasing_id())
)


In [ ]:
bronze_df.write.format("delta").mode("overwrite").partitionBy(
    "tpep_pickup_datetime"
).save(BRONZE_PATH)

# Partition columns are reported by Delta DESCRIBE DETAIL
print("Bronze written to:", BRONZE_PATH)
spark.sql(f"DESCRIBE DETAIL delta.`{BRONZE_PATH}`").select("partitionColumns").show(truncate=False)


## Silver Layer - Data Cleaning & Transformation


In [ ]:
silver_src = spark.read.format("delta").load(BRONZE_PATH)
print("Bronze / silver source row count:", silver_src.count())


In [ ]:
# Typical NYC TLC column names — adjust if your bronze schema differs
cleaned = (
    silver_src.filter(F.col("fare_amount") >= 0)
    .filter(F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0))
    .filter(F.col("trip_distance") > 0)
)


In [ ]:
# Parse pickup/dropoff if stored as strings; skip cast if already timestamp
pickup = F.to_timestamp("tpep_pickup_datetime")
dropoff = F.to_timestamp("tpep_dropoff_datetime")

silver_df = (
    cleaned.withColumn("trip_duration_sec", F.unix_timestamp(dropoff) - F.unix_timestamp(pickup))
    .withColumn(
        "speed_mph",
        F.when(F.col("trip_duration_sec") > 0, F.col("trip_distance") / (F.col("trip_duration_sec") / 3600.0)).otherwise(
            None
        ),
    )
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn("hour_of_day", F.hour(pickup))
    .withColumn("day_of_week", F.dayofweek(pickup))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False))
    .withColumn(
        "time_of_day",
        F.when((F.hour(pickup) >= 6) & (F.hour(pickup) < 12), "morning")
        .when((F.hour(pickup) >= 12) & (F.hour(pickup) < 17), "afternoon")
        .when((F.hour(pickup) >= 17) & (F.hour(pickup) < 22), "evening")
        .otherwise("night"),
    )
)


In [ ]:
# Merge/upsert into Silver on a business key (example: composite key)
silver_table = f"delta.`{SILVER_PATH}`"
if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    dt = DeltaTable.forPath(spark, SILVER_PATH)
    dt.alias("t").merge(
        silver_df.alias("s"),
        "t.VendorID = s.VendorID AND t.tpep_pickup_datetime = s.tpep_pickup_datetime AND t.PULocationID = s.PULocationID",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    silver_df.write.format("delta").mode("overwrite").save(SILVER_PATH)

spark.read.format("delta").load(SILVER_PATH).limit(3).show()


## Gold Layer - Aggregations


In [ ]:
s = spark.read.format("delta").load(SILVER_PATH)

hourly_stats = s.groupBy("hour_of_day").agg(
    F.count("*").alias("trip_count"),
    F.avg("fare_amount").alias("avg_fare"),
    F.avg("trip_distance").alias("avg_distance"),
    F.avg("trip_duration_sec").alias("avg_duration"),
    F.avg("speed_mph").alias("avg_speed"),
    F.sum("fare_amount").alias("total_revenue"),
).withColumn("event_timestamp", F.current_timestamp())


In [ ]:
from pyspark.sql.window import Window as W

daily = s.groupBy(F.to_date("tpep_pickup_datetime").alias("trip_date")).agg(
    F.count("*").alias("daily_trip_count"),
    F.avg("fare_amount").alias("daily_avg_fare"),
    F.sum("fare_amount").alias("daily_total_revenue"),
)

hourly_by_day = s.groupBy(F.to_date("tpep_pickup_datetime").alias("trip_date"), "hour_of_day").agg(
    F.count("*").alias("cnt")
)
w = W.partitionBy("trip_date").orderBy(F.desc("cnt"))
daily_stats = (
    hourly_by_day.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("trip_date", F.col("hour_of_day").alias("peak_hour"))
    .join(daily, "trip_date", "inner")
    .withColumn("event_timestamp", F.current_timestamp())
)


In [ ]:
zone_stats = (
    s.groupBy(F.col("PULocationID").alias("zone_id"))
    .agg(
        F.avg("fare_amount").alias("zone_avg_fare"),
        F.count("*").alias("zone_trip_count"),
        F.avg("trip_distance").alias("zone_avg_distance"),
    )
    .withColumn("event_timestamp", F.current_timestamp())
)


In [ ]:
hourly_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}hourly_stats/")
daily_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}daily_stats/")
zone_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}zone_stats/")
print("Gold tables written.")


In [ ]:
hpd = hourly_stats.orderBy("hour_of_day").toPandas()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(hpd["hour_of_day"], hpd["trip_count"])
axes[0].set_title("Trips by hour")
axes[0].set_xlabel("Hour of day")

dow = (
    s.groupBy("day_of_week")
    .agg(F.avg("fare_amount").alias("avg_fare"))
    .orderBy("day_of_week")
    .toPandas()
)
axes[1].plot(dow["day_of_week"], dow["avg_fare"], marker="o")
axes[1].set_title("Avg fare by day of week (1=Sun)")
axes[1].set_xlabel("day_of_week")
plt.tight_layout()
plt.show()


## Summary

**Data quality metrics (examples to track):**
- Row counts at each layer (bronze ≥ silver after filters).
- Null rates on keys (`VendorID`, `PULocationID`, timestamps).
- Share of rows removed by cleaning (negative fare, zero distance, invalid passengers).
- Distribution checks on `speed_mph` and `fare_per_mile` for outliers.
